# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
!git clone https://github.com/Ebrahim827/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

import pandas as pd
import numpy as np
import json, os

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape[0], "rows")

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 234, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 234 (delta 97), reused 59 (delta 59), pack-reused 111 (from 3)
Receiving objects: 100% (234/234), 1.89 MiB | 3.38 MiB/s, done.
Resolving deltas: 100% (112/112), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Loaded: 30000 rows


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This turns everything I checked in Weeks 4-6 into an actual to-do list. Instead of one messy score, I'm using three simple, separate flags — each one is its own reason a page needs attention:

Stale but visible → the page hasn't been touched in a long time, but people still find it. Action: refresh it.
CTR underperforming for its position → the page ranks well but way fewer people click on it than similar pages in the same rank. Action: fix the title/snippet.
Strong performer → the page is fresh, ranks well, and gets a lot of traffic. Action: don't touch it, just keep an eye on it.
Anything left over gets no action — not every page needs attention right now.

Important honest note: I'm ranking these using my simple Week-4 rule, not my Week-5 ML model. Why? Because in Week 6, when I tested the model the honest way (keeping each client's pages fully separate between training and testing), it only scored 0.55 — barely better than just guessing (0.559 base rate). The simple rule is the more trustworthy option right now, so that's what drives this playbook.

In [5]:
STALE_DAYS = 180
VISIBLE_IMPRESSIONS = 500

# CTR benchmark: what's "normal" CTR for each position tier (reused from Week 4's signal check)
tier_median_ctr = df.groupby("position_tier")["ctr"].median()
df["_tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)

df["stale"] = (df["days_since_last_update"] >= STALE_DAYS)
df["visible"] = (df["impressions_90d"] >= VISIBLE_IMPRESSIONS)
df["ctr_underperforming"] = (
    (df["ctr"] < 0.5 * df["_tier_median_ctr"]) & df["visible"] &
    df["position_tier"].isin(["striking", "top_3", "page_1"])
)
df["strong_performer"] = (
    (df["avg_position"] <= 10) &
    (df["impressions_90d"] >= df["impressions_90d"].quantile(0.75)) &
    (df["days_since_last_update"] < 90)
)

def assign_reason(row):
    if row["stale"] and row["visible"]:
        return "stale_but_visible", "refresh_review"
    elif row["ctr_underperforming"]:
        return "ctr_underperforming_for_position", "ctr_fix_review"
    elif row["strong_performer"]:
        return "strong_performer", "protect_monitor"
    else:
        return "no_flag", "no_action"

reasons = df.apply(assign_reason, axis=1, result_type="expand")
df["reason_code"] = reasons[0]
df["action"] = reasons[1]

print("How many pages got each label:")
print(df["reason_code"].value_counts())

queue = df[df["reason_code"] != "no_flag"].sort_values("impressions_90d", ascending=False)
out_cols = ["content_id", "client_id", "reason_code", "action",
            "days_since_last_update", "impressions_90d", "ctr", "avg_position"]
print("\nTop of the ranked queue:")
print(queue[out_cols].head(15))

How many pages got each label:
reason_code
no_flag                             25564
strong_performer                     2438
ctr_underperforming_for_position     1981
stale_but_visible                      17
Name: count, dtype: int64

Top of the ranked queue:
                 content_id          client_id  \
17812  content_aaef01a50def  client_19581e27de   
26844  content_8c19996aa890  client_4e07408562   
21819  content_4c36c775b818  client_4e07408562   
29879  content_1a9e894be2e2  client_19581e27de   
18870  content_db5989a78dd3  client_4e07408562   
14090  content_44e481c8f55b  client_19581e27de   
3394   content_36ff89c8214e  client_19581e27de   
16811  content_8e7ba84a972b  client_7f2253d7e2   
14234  content_89e84d699e9e  client_349c41201b   
7678   content_8451fc6f034d  client_d029fa3a95   
15250  content_aa4baf490b43  client_349c41201b   
27478  content_008fb02c46cb  client_349c41201b   
6903   content_c84a0ab98e90  client_f369cb89fc   
10741  content_07e0b9af8b1a  client_b

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who this is for:** a content/SEO team member doing a manual review pass — not a fully automatic system. Think of it as a smart to-do list, not a robot that fixes pages by itself.

**What it can honestly say:** in this dataset, pages that are old-but-still-visible, or ranking well-but-underclicked, showed a pattern worth checking. That's it — a pattern in this data, not a proven cause-and-effect rule. I can't say "refreshing a page will definitely increase its traffic" — I only have a snapshot of data, not a real before/after experiment. **The honest way to say it is: "these pages look worth reviewing first, because of X."**

**Limits:**

This can't tell you why a page is declining — only that it matches a pattern.
It doesn't know about outside factors — a competitor launching something, Google changing its algorithm, a season ending, etc.
The thresholds I used (180 days, 500 impressions) are reasonable guesses based on what I saw in my signal checks, not perfectly tuned numbers.
This is a snapshot from one point in time, not something that watches pages continuously.

In [6]:
print("Section 2 done.See text above for intended use and limits.")

Section 2 done.See text above for intended use and limits.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any flagged page, a person should check:**

Read the actual page — does the flag make sense, or is there an obvious reason (like a seasonal topic) that explains it?
For "CTR fix" flags: look at the actual title/snippet in search results — sometimes it's already fine and the number is just noisy.
For "refresh" flags: confirm the page still matches the business's current products/services before spending time rewriting it.
For "strong performer" flags: just because it's doing well doesn't mean it needs zero attention forever — recheck periodically.

**What should NEVER be automated:**

Never auto-publish a rewritten page without a human reading it first.
Never auto-delete a page just because it's flagged "no action" or low-performing.
Never treat any of these flags as a guarantee, or promise a client a specific traffic increase based on this list.
Never use this list for pages tied to legal, medical, financial, or safety topics without extra review — those need expert judgment, not a traffic-pattern flag.
Never fully trust the "CTR fix" flag on a page with very little traffic — small numbers can look extreme just by chance.

In [7]:
print("Section 3 done. See above text cell for the human-review checklist and no-go list.")

Section 3 done. See above text cell for the human-review checklist and no-go list.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**What would tell me this playbook has gone stale and needs a refresh?**

If the overall "base rate" (how many pages are naturally declining) shifts a lot from what I measured (0.559) — that means the whole dataset's behavior changed, and old thresholds may not fit anymore.

If a big batch of pages get refreshed all at once — that changes the "days since update" numbers for a lot of pages at the same time, so the rule should be re-run after any big refresh push.

If the number of flagged pages suddenly jumps or drops a lot compared to normal — that's a sign something upstream changed (tracking broke, a new content type was added, etc.), not necessarily that more pages actually need help.

On a regular basis (I'd suggest monthly), rerun this whole notebook fresh rather than reusing old results — the queue is a snapshot, not a live feed.

In [8]:
print("Section 4 done — see markdown above for monitoring/retrain triggers.")

Section 4 done — see markdown above for monitoring/retrain triggers.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

This part saves the final results so next week's paper can use them as real proof, instead of just typing claims with nothing behind them.
The actual list of flagged pages (the CSV file) doesn't get saved into GitHub — that's on purpose, since raw data files aren't allowed in the repo. But the notebook can always make that file again by just running it.
The numbers file (the JSON) does get saved into GitHub. That one is like a receipt — it proves where the paper's numbers actually came from, so anyone checking the work later can see it's real.

In [9]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

queue[out_cols].to_csv("work/outputs/w07_action_playbook_queue.csv", index=False)
print("Wrote ranked queue:", len(queue), "actionable rows out of", len(df), "total")

metrics = {
    "n_total_rows": int(len(df)),
    "n_flagged": int(len(queue)),
    "reason_code_counts": df["reason_code"].value_counts().to_dict(),
    "week6_honest_precision_at_20": 0.55,
    "week6_base_rate": 0.559,
    "note": ("Ranking uses the Week-4 rule baseline, not the Week-5 ML model, because the "
             "honest grouped-split test in Week 6 showed the model barely beat the base rate "
             "(0.55 vs 0.559) on pages from clients it hadn't seen before.")
}
with open("work/outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote metrics JSON — this file IS committed to git as your paper's receipts.")

Wrote ranked queue: 4436 actionable rows out of 30000 total
Wrote metrics JSON — this file IS committed to git as your paper's receipts.


## Self-check

Before you submit, confirm each line honestly:

- [ ✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✔] No client names, URLs, or private queries anywhere
- [✔ ] My claims use careful words: observed, measured, directional, decision-support
- [✔ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.